In [ ]:
# BLOCO 1: Instalações que Exigem Reinício

# 1. Downgrade preventivo do Numpy (para evitar conflitos com AV)
!pip install "numpy<2.0"

# 2. Instala AV (biblioteca de decodificação) na versão estável (evita o Silent Hang do MP3)
!pip install "av==15.0.0"

print("\n-------------------------------------------------------------------------")
print("✅ INSTALAÇÕES INICIAIS CONCLUÍDAS.")
print("🔴 AÇÃO CRÍTICA: Você DEVE REINICIAR o ambiente agora para estabilizar o numpy.")
print("Vá em Runtime (Ambiente de Execução) > Restart session (Reiniciar Sessão) ou clique no botão de reinício.")
print("-------------------------------------------------------------------------")

In [ ]:
# BLOCO 2: Instalações Finais (Execute SOMENTE APÓS o REINÍCIO da Sessão do Bloco 1)

# 1. Força a reinstalação do PyTorch + TorchVision sincronizados
#    A versão 2.3.1 é compatível com o ambiente CUDA 11.8 da GPU T4 do Colab.
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118

# 2. Instala WhisperX e dependências 
!pip install git+https://github.com/m-bain/whisperX.git --no-deps
!pip install ffmpeg-python pandas scipy "pyannote.audio==3.3.2" ctranslate2 faster-whisper

print("\n✅ INSTALAÇÃO E CONFIGURAÇÃO COMPLETAS. O ambiente está pronto.")

In [ ]:
# BLOCO 3: MONTAGEM DO DRIVE E CONFIGURAÇÃO DE SESSÃO
from google.colab import drive
import os

drive.mount('/content/drive')

# ----------------------------------------------------------------------
# EDITE APENAS ESTAS VARIÁVEIS PARA CADA NOVA SESSÃO:
# ----------------------------------------------------------------------

# O ID da mesa (e.g., "THoL", "ID", "DiT") e número da sessão (e.g., "01", "10", "002")
ID_MESA = "ID"
NUM_SESSAO = "10"

# O ID da sessão atual (e.g., "THoL01", "ID56", "DiT10")
ID_SESSAO = f"{ID_MESA}{NUM_SESSAO}"

# O caminho base da sua pasta de projeto no Google Drive
PASTA_PROJETO = "/content/drive/MyDrive/rpg-transcription"

# ----------------------------------------------------------------------
# DEFINE O GLOSSÁRIO (Para Whisper e Gemini)
# ----------------------------------------------------------------------

# Cria a pasta utils se não existir
if not os.path.exists("utils"):
    os.makedirs("utils")

# Baixa o arquivo para dentro da pasta utils
github_url = "https://raw.githubusercontent.com/ThaisMosken/rpg_transcription/main/utils/glossary_manager.py"
!wget -O utils/glossary_manager.py {github_url}

# Importa a classe do módulo
from utils.glossary_manager import GlossaryManager

# Inicializa o gerenciador
manager = GlossaryManager(ID_MESA)

# Obtém o conteúdo pronto para a transcrição e o prompt
glossario_contexto = manager.get_full_glossary()

GLOSSARIO_NOMES = [
    linha.strip("- *").strip() 
    for linha in glossario_contexto.split('\n') 
    if linha.strip() and not linha.startswith('#')
]

print(f"✅ Glossário para '{ID_MESA}' carregado com sucesso!")

# ----------------------------------------------------------------------
# VARIÁVEIS DERIVADAS (Não precisam ser editadas)
# ----------------------------------------------------------------------

# Arquivo de entrada para a transcrição
ARQUIVO_WAV_ENTRADA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}w.wav")

# Arquivo TXT de saída da transcrição (será lido pelo Gemini)
ARQUIVO_TXT_SAIDA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}_transcricao_final.txt")

# Arquivo Markdown (Crônica) de saída do Gemini
ARQUIVO_CRONICA_SAIDA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}_cronica.md")

print("Configuração de sessão concluída:")
print(f"  ID da Sessão: {ID_SESSAO}")
print(f"  Entrada WAV: {ARQUIVO_WAV_ENTRADA}")
print(f"  Saída TXT: {ARQUIVO_TXT_SAIDA}")
print(f"  Saída Crônica: {ARQUIVO_CRONICA_SAIDA}")

In [ ]:
# BLOCO 4: Execução da Transcrição (Faster-Whisper)
import os

# 1. Clone o repositório (se ainda não existir)
if not os.path.exists('rpg_transcription'):
    !git clone https://github.com/ThaisMosken/rpg_transcription.git

# 2. Adicione a pasta do projeto ao 'path' do sistema para o Python encontrar o SRC
import sys
sys.path.append('/content/rpg_transcription')

# No Colab, agora você apenas chama a função do seu repositório
from src.transcription_processor import executar_transcricao

# Executa a função passando as variáveis que você já definiu nos blocos anteriores
executar_transcricao(
    arquivo_entrada=ARQUIVO_WAV_ENTRADA,
    arquivo_saida=ARQUIVO_TXT_SAIDA,
    glossario_nomes=GLOSSARIO_NOMES,
    dispositivo="cuda" # ou "cpu"
)

In [ ]:
# BLOCO 5: Geração da Crônica
from google.colab import userdata
from src.chronicler import gerar_cronica_gemini

# 1. Tenta pegar a chave do Secret do Colab
try:
    CHAVE_GEMINI = userdata.get('GEMINI_API_KEY')
except:
    CHAVE_GEMINI = None
    print("ERRO: Secret 'GEMINI_API_KEY' não encontrado.")

# 2. Define o modelo
MODELO_ESCOLHIDO = 'gemini-2.5-flash'

# 3. Executa a função importada
if CHAVE_GEMINI:
    gerar_cronica_gemini(
        api_key=CHAVE_GEMINI,
        caminho_transcricao=ARQUIVO_TXT_SAIDA,
        caminho_saida=ARQUIVO_CRONICA_SAIDA,
        glossario_contexto=glossario_contexto,
        modelo=MODELO_ESCOLHIDO
    )